# SetStop-KGQA — Notebook 02
## WebQSP state generation and symbolic baselines

**Runtime:** CPU only (Colab: Runtime → Change runtime type → Hardware accelerator → None).

This notebook uses the already extracted NSM WebQSP data. It creates the filtered manifest, leakage-safe frontier supervision states, integrity reports, and symbolic baseline results. Gold answers are used only for offline labels and oracle answer recognition; they are never written into model input text.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import collections, hashlib, json, math, random, re, statistics, time
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")
DATA_DIR = ROOT / "data/extracted/webqsp"
PROCESSED = ROOT / "data/processed"
OUTPUTS = ROOT / "outputs/preprocessing_v1"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

# First run: keep PILOT=True. After it succeeds, set False and Run all again.
PILOT = True
PILOT_QUESTIONS_PER_SPLIT = 20
MAX_ANSWERS = 50
MAX_HOPS = 3
MAX_STEPS = 16
MAX_CANDIDATES_PER_STATE = 64
MAX_STATES_PER_QUESTION = 24

required = ["train_simple.json","dev_simple.json","test_simple.json","entities.txt","relations.txt"]
for name in required:
    assert (DATA_DIR/name).exists(), f"Missing: {DATA_DIR/name}"
print("CPU preprocessing configuration ready")
print("Mode:", "PILOT" if PILOT else "FULL")

Mounted at /content/drive
CPU preprocessing configuration ready
Mode: PILOT


In [2]:
def read_vocab(path):
    return [x.rstrip("\n") for x in open(path, encoding="utf-8")]

ENTITY_NAMES = read_vocab(DATA_DIR / "entities.txt")
RELATION_NAMES = read_vocab(DATA_DIR / "relations.txt")
ENTITY_TO_ID = {x:i for i,x in enumerate(ENTITY_NAMES)}

def iter_jsonl(path):
    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if line.strip():
                try: yield json.loads(line)
                except Exception as e: raise ValueError(f"Bad JSON at {path}:{line_no}") from e

def entity_index(x):
    if isinstance(x, (int, np.integer)): return int(x)
    if isinstance(x, str): return ENTITY_TO_ID.get(x)
    if isinstance(x, dict):
        for k in ("kb_id","id","entity"):
            if k in x: return entity_index(x[k])
    return None

def get_question(ex):
    for k in ("question","text","query"):
        if isinstance(ex.get(k), str): return ex[k]
    raise KeyError("No question/text/query field")

def get_topic_ids(ex):
    raw = ex.get("entities", ex.get("topic_entities", []))
    out = {entity_index(x) for x in raw}
    return {x for x in out if x is not None}

def get_gold_ids(ex):
    out = {entity_index(x) for x in ex.get("answers", [])}
    return {x for x in out if x is not None}

def get_graph(ex):
    sg = ex.get("subgraph", {})
    entities = {entity_index(x) for x in sg.get("entities", [])}
    entities.discard(None)
    raw_edges = sg.get("tuples", sg.get("triples", []))
    edges=[]
    for row in raw_edges:
        if isinstance(row, dict):
            s=entity_index(row.get("subject",row.get("s"))); r=row.get("relation",row.get("r")); o=entity_index(row.get("object",row.get("o")))
        else:
            if len(row)!=3: continue
            s,r,o=row; s=entity_index(s); o=entity_index(o)
        if s is None or o is None: continue
        if isinstance(r, str):
            try: r=RELATION_NAMES.index(r)
            except ValueError: continue
        edges.append((int(s),int(r),int(o)))
        entities.update((int(s),int(o)))
    return entities, edges

sample = next(iter_jsonl(DATA_DIR / "train_simple.json"))
print("Top-level keys:", sorted(sample))
print("Question:", get_question(sample)[:150])
print("Topics/gold:", len(get_topic_ids(sample)), len(get_gold_ids(sample)))
print("Graph entities/edges:", *(len(x) for x in get_graph(sample)))
assert get_topic_ids(sample), "Parser found no topic entities"
assert get_graph(sample)[1], "Parser found no subgraph edges"
print("Schema check passed")

Top-level keys: ['answers', 'entities', 'id', 'question', 'subgraph']
Question: what is the name of justin bieber brother
Topics/gold: 1 1
Graph entities/edges: 2000 9088
Schema check passed


In [3]:
def stable_qid(split, ex, line_no):
    for k in ("id","qid","question_id"):
        if k in ex: return str(ex[k])
    h=hashlib.sha1(get_question(ex).encode()).hexdigest()[:12]
    return f"{split}_{line_no}_{h}"

manifest=[]; selected={}
for split in ("train","dev","test"):
    chosen=[]
    for line_no, ex in enumerate(iter_jsonl(DATA_DIR/f"{split}_simple.json"),1):
        qid=stable_qid(split,ex,line_no); gold=get_gold_ids(ex); topics=get_topic_ids(ex); nodes,edges=get_graph(ex)
        usable = 2 <= len(gold) <= MAX_ANSWERS and gold.issubset(nodes) and bool(topics) and bool(edges)
        row={"qid":qid,"split":split,"line_no":line_no,"n_gold":len(gold),"n_topics":len(topics),"n_nodes":len(nodes),"n_edges":len(edges),"usable":usable}
        manifest.append(row)
        if usable and (not PILOT or len(chosen)<PILOT_QUESTIONS_PER_SPLIT): chosen.append((qid,ex))
    selected[split]=chosen
    print(split,"selected",len(chosen))

suffix="pilot" if PILOT else "full"
manifest_path=PROCESSED/f"webqsp_manifest_v1_{suffix}.json"
manifest_path.write_text(json.dumps({"config":{"seed":SEED,"max_answers":MAX_ANSWERS,"max_hops":MAX_HOPS},"rows":manifest},indent=2),encoding="utf-8")
print("Saved",manifest_path)

train selected 20
dev selected 20
test selected 20
Saved /content/drive/MyDrive/SetStop_KGQA/data/processed/webqsp_manifest_v1_pilot.json


In [4]:
def adjacency(edges):
    adj=collections.defaultdict(list)
    for s,r,o in edges:
        # NSM relations already contain inverse relations; preserve released direction only.
        adj[s].append((r,o))
    for s in adj: adj[s].sort()
    return adj

def reachable_gold(start, adj, gold, blocked, depth):
    seen=set(blocked); seen.discard(start); layer={start}; found=set()
    for _ in range(depth+1):
        found |= layer & gold
        nxt=set()
        for u in layer:
            for _,v in adj.get(u,[]):
                if v not in seen: nxt.add(v)
        seen |= layer; layer=nxt
        if not layer: break
    return found

def rel_name(r): return RELATION_NAMES[r] if 0 <= r < len(RELATION_NAMES) else f"rel_{r}"
def ent_name(e): return ENTITY_NAMES[e] if 0 <= e < len(ENTITY_NAMES) else f"entity_{e}"

def frontier(visited, adj):
    # One candidate per directed boundary edge; path identity is retained.
    out=[]
    for s in sorted(visited):
        for r,o in adj.get(s,[]):
            if o not in visited: out.append((s,r,o))
    return out

def candidate_record(c, adj, gold, found, visited, remaining_hops, question):
    s,r,o=c
    new=reachable_gold(o,adj,gold-found,visited,remaining_hops)
    gain=len(new)
    gain_class="3_plus" if gain>=3 else str(gain)
    # model_text contains state-observable information only; never gold IDs or gain.
    model_text=(f"Question: {question}\nCurrent found count: {len(found)}\n"
                f"Visited nodes: {len(visited)}\nCandidate relation: {rel_name(r)}\n"
                f"Candidate source: {ent_name(s)}\nCandidate target: {ent_name(o)}")
    return {"source":s,"relation":r,"target":o,"relation_text":rel_name(r),"gain":gain,"gain_class":gain_class,"model_text":model_text}

def build_states(qid, ex):
    q=get_question(ex); gold=get_gold_ids(ex); topics=get_topic_ids(ex); _,edges=get_graph(ex); adj=adjacency(edges)
    visited=set(topics); found=visited & gold; history=[]; rows=[]
    rng=random.Random(SEED + int(hashlib.sha1(qid.encode()).hexdigest()[:8],16))
    for step in range(MAX_STEPS):
        raw=frontier(visited,adj)
        if not raw: break
        remaining=max(0,MAX_HOPS-min(step,MAX_HOPS-1)-1)
        candidates=[candidate_record(c,adj,gold,found,visited,remaining,q) for c in raw]
        # Retain positives, hard zero-gain candidates, and a deterministic random sample.
        candidates.sort(key=lambda x:(-x["gain"],x["relation"],x["target"]))
        positives=[x for x in candidates if x["gain"]>0]
        zeros=[x for x in candidates if x["gain"]==0]; rng.shuffle(zeros)
        kept=(positives+zeros)[:MAX_CANDIDATES_PER_STATE]
        stop_label = int(max((x["gain"] for x in candidates),default=0)==0)
        rows.append({"qid":qid,"question":q,"step":step,"found_count":len(found),"visited_count":len(visited),"frontier_count":len(raw),"remaining_hops":remaining,"stop_label":stop_label,"candidate_count":len(kept),"candidates":kept,"history_relations":[rel_name(x) for x in history[-8:]]})
        if len(rows)>=MAX_STATES_PER_QUESTION: break
        # Offline trajectory policy: choose a max-gain edge; once complete, add one zero-gain step.
        best_gain=max(x["gain"] for x in candidates)
        pool=[x for x in candidates if x["gain"]==best_gain]
        chosen=pool[rng.randrange(len(pool))]
        visited.add(chosen["target"]); history.append(chosen["relation"]); found=visited & gold
        if found==gold:
            # The next stored state becomes a complete STOP example if a frontier remains.
            raw2=frontier(visited,adj)
            if raw2:
                cs=[candidate_record(c,adj,gold,found,visited,0,q) for c in raw2[:MAX_CANDIDATES_PER_STATE]]
                rows.append({"qid":qid,"question":q,"step":step+1,"found_count":len(found),"visited_count":len(visited),"frontier_count":len(raw2),"remaining_hops":0,"stop_label":1,"candidate_count":len(cs),"candidates":cs,"history_relations":[rel_name(x) for x in history[-8:]]})
            break
    return rows

state_paths={}; stats={}
for split, items in selected.items():
    path=PROCESSED/f"webqsp_{split}_states_v1_{suffix}.jsonl"; state_paths[split]=path
    nstates=ncands=npos=nstop=0
    with open(path,"w",encoding="utf-8") as out:
        for i,(qid,ex) in enumerate(items,1):
            for row in build_states(qid,ex):
                out.write(json.dumps(row,ensure_ascii=False)+"\n")
                nstates+=1; nstop+=row["stop_label"]; ncands+=row["candidate_count"]
                npos+=sum(c["gain"]>0 for c in row["candidates"])
            if i%100==0: print(split,i,"questions")
    stats[split]={"questions":len(items),"states":nstates,"candidates":ncands,"positive_candidates":npos,"stop_states":nstop}
    print(split,stats[split],"->",path)
(OUTPUTS/f"state_generation_summary_v1_{suffix}.json").write_text(json.dumps(stats,indent=2),encoding="utf-8")

train {'questions': 20, 'states': 189, 'candidates': 12096, 'positive_candidates': 2085, 'stop_states': 63} -> /content/drive/MyDrive/SetStop_KGQA/data/processed/webqsp_train_states_v1_pilot.jsonl
dev {'questions': 20, 'states': 181, 'candidates': 11189, 'positive_candidates': 2360, 'stop_states': 65} -> /content/drive/MyDrive/SetStop_KGQA/data/processed/webqsp_dev_states_v1_pilot.jsonl
test {'questions': 20, 'states': 178, 'candidates': 11348, 'positive_candidates': 2897, 'stop_states': 41} -> /content/drive/MyDrive/SetStop_KGQA/data/processed/webqsp_test_states_v1_pilot.jsonl


413

In [6]:
# Corrected integrity and target-leakage audit

audit = {}

# Only detect explicit fields that could reveal training targets.
# Do not search ordinary words such as "gold" or "gain" inside questions.
forbidden_patterns = [
    r"(?im)^\s*gold(?:\s+answers?|\s+entities?|\s+ids?)?\s*:",
    r"(?im)^\s*gain\s*:",
    r"(?im)^\s*gain_class\s*:",
    r"(?im)^\s*stop_label\s*:",
    r"(?im)^\s*target_label\s*:",
    r"(?im)^\s*oracle_gain\s*:",
]

for split, path in state_paths.items():
    labels = collections.Counter()
    stop_labels = collections.Counter()

    number_of_states = 0
    number_of_candidates = 0
    leakage_examples = []

    for row in iter_jsonl(path):
        number_of_states += 1
        stop_labels[str(row["stop_label"])] += 1

        gains = [
            candidate["gain"]
            for candidate in row["candidates"]
        ]

        # Confirm that stop_label agrees with candidate gains.
        expected_stop_label = int(
            max(gains, default=0) == 0
        )

        if row["stop_label"] != expected_stop_label:
            raise AssertionError(
                f"Inconsistent stop label for {row['qid']}: "
                f"stored={row['stop_label']}, "
                f"expected={expected_stop_label}"
            )

        for candidate in row["candidates"]:
            number_of_candidates += 1
            labels[candidate["gain_class"]] += 1

            model_text = candidate.get("model_text", "")

            matched_patterns = [
                pattern
                for pattern in forbidden_patterns
                if re.search(pattern, model_text)
            ]

            if matched_patterns:
                leakage_examples.append({
                    "qid": row["qid"],
                    "patterns": matched_patterns,
                    "model_text": model_text,
                })

    assert number_of_states > 0, f"No states found in {path}"

    audit[split] = {
        "states": number_of_states,
        "candidates": number_of_candidates,
        "gain_classes": dict(labels),
        "stop_labels": dict(stop_labels),
        "leakage_flags": len(leakage_examples),
        "leakage_examples": leakage_examples[:5],
    }

    if leakage_examples:
        print(f"\nPotential leakage examples in {split}:")
        for example in leakage_examples[:5]:
            print(json.dumps(example, indent=2, ensure_ascii=False))

assert all(
    result["leakage_flags"] == 0
    for result in audit.values()
), "Explicit target fields were found in model_text"

print(json.dumps(audit, indent=2, ensure_ascii=False))

audit_path = (
    OUTPUTS
    / f"state_integrity_audit_v1_{suffix}.json"
)

audit_path.write_text(
    json.dumps(audit, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nIntegrity audit passed.")
print("Saved:", audit_path)

{
  "train": {
    "states": 189,
    "candidates": 12096,
    "gain_classes": {
      "2": 251,
      "1": 1647,
      "0": 10011,
      "3_plus": 187
    },
    "stop_labels": {
      "0": 126,
      "1": 63
    },
    "leakage_flags": 0,
    "leakage_examples": []
  },
  "dev": {
    "states": 181,
    "candidates": 11189,
    "gain_classes": {
      "2": 170,
      "1": 1902,
      "0": 8829,
      "3_plus": 288
    },
    "stop_labels": {
      "0": 116,
      "1": 65
    },
    "leakage_flags": 0,
    "leakage_examples": []
  },
  "test": {
    "states": 178,
    "candidates": 11348,
    "gain_classes": {
      "1": 2463,
      "0": 8451,
      "3_plus": 217,
      "2": 217
    },
    "stop_labels": {
      "0": 137,
      "1": 41
    },
    "leakage_flags": 0,
    "leakage_examples": []
  }
}

Integrity audit passed.
Saved: /content/drive/MyDrive/SetStop_KGQA/outputs/preprocessing_v1/state_integrity_audit_v1_pilot.json


In [7]:
# Symbolic oracle-recognition baselines. Selection policies do not see gold except oracle_gain.
def simulate(ex, policy, budget=16, patience_n=None):
    q=get_question(ex); gold=get_gold_ids(ex); topics=get_topic_ids(ex); _,edges=get_graph(ex); adj=adjacency(edges)
    visited=set(topics); found=visited&gold; no_gain=0; expansions=0
    rng=random.Random(SEED)
    while expansions<budget:
        fs=frontier(visited,adj)
        if not fs: break
        if policy=="fifo": chosen=fs[0]
        elif policy=="random": chosen=fs[rng.randrange(len(fs))]
        elif policy=="relation_overlap":
            toks=set(re.findall(r"[a-z0-9]+",q.lower()))
            chosen=max(fs,key=lambda x:(len(toks & set(re.findall(r"[a-z0-9]+",rel_name(x[1]).lower()))),-x[1],-x[2]))
        elif policy=="oracle_gain":
            chosen=max(fs,key=lambda c:len(reachable_gold(c[2],adj,gold-found,visited,MAX_HOPS)))
            if len(reachable_gold(chosen[2],adj,gold-found,visited,MAX_HOPS))==0: break
        else: raise ValueError(policy)
        before=len(found); visited.add(chosen[2]); found=visited&gold; expansions+=1
        no_gain = no_gain+1 if len(found)==before else 0
        if found==gold: break
        if patience_n is not None and no_gain>=patience_n: break
    p=len(found)/max(1,len(visited&gold)) if found else 0.0 # oracle-recognition precision is 1 when nonempty
    recall=len(found)/len(gold); f1=0 if not found else 2*recall/(1+recall)
    return {"f1":f1,"recall":recall,"exact":int(found==gold),"expansions":expansions,"premature":int(found!=gold)}

configs=[("fifo_b8","fifo",8,None),("fifo_b16","fifo",16,None),("patience1","fifo",16,1),("patience2","fifo",16,2),("relation_b16","relation_overlap",16,None),("oracle_gain","oracle_gain",16,None)]
results={}
for split in ("dev","test"):
    results[split]={}
    for name,policy,budget,pat in configs:
        vals=[simulate(ex,policy,budget,pat) for _,ex in selected[split]]
        results[split][name]={k:float(np.mean([v[k] for v in vals])) for k in vals[0]}
print(json.dumps(results,indent=2))
baseline_path=OUTPUTS/f"symbolic_baselines_v1_{suffix}.json"
baseline_path.write_text(json.dumps({"mode":suffix,"config":configs,"results":results},indent=2),encoding="utf-8")
print("Saved",baseline_path)

{
  "dev": {
    "fifo_b8": {
      "f1": 0.08333333333333333,
      "recall": 0.075,
      "exact": 0.05,
      "expansions": 7.85,
      "premature": 0.95
    },
    "fifo_b16": {
      "f1": 0.15,
      "recall": 0.125,
      "exact": 0.05,
      "expansions": 15.45,
      "premature": 0.95
    },
    "patience1": {
      "f1": 0.0,
      "recall": 0.0,
      "exact": 0.0,
      "expansions": 1.0,
      "premature": 1.0
    },
    "patience2": {
      "f1": 0.0,
      "recall": 0.0,
      "exact": 0.0,
      "expansions": 2.0,
      "premature": 1.0
    },
    "relation_b16": {
      "f1": 0.5271186440677966,
      "recall": 0.5186046511627908,
      "exact": 0.5,
      "expansions": 12.15,
      "premature": 0.5
    },
    "oracle_gain": {
      "f1": 0.6548421525671273,
      "recall": 0.6393680017226528,
      "exact": 0.6,
      "expansions": 12.1,
      "premature": 0.4
    }
  },
  "test": {
    "fifo_b8": {
      "f1": 0.0,
      "recall": 0.0,
      "exact": 0.0,
      "expa

In [8]:
# Final inventory and SHA-256 checksums
artifacts=[manifest_path,*state_paths.values(),OUTPUTS/f"state_generation_summary_v1_{suffix}.json",OUTPUTS/f"state_integrity_audit_v1_{suffix}.json",baseline_path]
inventory=[]
for p in artifacts:
    h=hashlib.sha256(p.read_bytes()).hexdigest()
    inventory.append({"path":str(p),"bytes":p.stat().st_size,"sha256":h})
    print(p.name, f"{p.stat().st_size/1024**2:.2f} MB", h[:16])
inv_path=OUTPUTS/f"artifact_inventory_v1_{suffix}.json"
inv_path.write_text(json.dumps(inventory,indent=2),encoding="utf-8")
print("DONE. Next: set PILOT=False and Run all. Upload/share the summary, audit, baselines, and inventory JSON files for analysis.")

webqsp_manifest_v1_pilot.json 0.88 MB 791f797e6b3ab1ce
webqsp_train_states_v1_pilot.jsonl 4.14 MB 85df7eff551f3409
webqsp_dev_states_v1_pilot.jsonl 3.89 MB 1bc6822ed618fb92
webqsp_test_states_v1_pilot.jsonl 4.02 MB 60ec0b97a8597ba9
state_generation_summary_v1_pilot.json 0.00 MB 37e521df4c2e0a09
state_integrity_audit_v1_pilot.json 0.00 MB bb85051d0935ea49
symbolic_baselines_v1_pilot.json 0.00 MB bcf42535ff0cc6ec
DONE. Next: set PILOT=False and Run all. Upload/share the summary, audit, baselines, and inventory JSON files for analysis.
